# 2025 Strength of Schedule: Opponent-Adjusted EPA

For each NFL team, compute two SOS scores based on their 2025 opponents' 2025 performance:
- **Offensive SOS** — average adjusted defensive EPA of opponents (lower = harder for your offense)
- **Defensive SOS** — average adjusted offensive EPA of opponents (higher = harder for your defense)

EPA values are opponent-adjusted: each team's unit efficiency is credited/discounted based on the quality of opponents they faced in 2025.

In [7]:
# Run this cell only if packages aren't installed yet
# !pip install nfl_data_py pandas -q

In [5]:
import nfl_data_py as nfl
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## Step 1: Load 2025 Play-by-Play Data

Pull all 2025 plays (regular season + playoffs). We filter to standard pass and run plays only — no kickoffs, two-point conversions, or special teams.

In [6]:
pbp = nfl.import_pbp_data([2025])

# Filter to pass and run plays, exclude two-point attempts
pbp_filtered = pbp[
    (pbp['play_type'].isin(['pass', 'run'])) &
    (pbp['two_point_attempt'] == 0) &
    (pbp['epa'].notna()) &
    (pbp['posteam'].notna())
].copy()

print(f"Total plays: {len(pbp_filtered):,}")
print(f"Weeks covered: {pbp_filtered['week'].min()} to {pbp_filtered['week'].max()}")
print(f"Teams: {pbp_filtered['posteam'].nunique()}")

<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1123)>
Data not available for 2025


KeyError: 'play_type'

## Step 2: Compute Raw EPA/Play by Unit

- **Offensive EPA/play**: from the offense's perspective (`posteam`)
- **Defensive EPA/play**: from the defense's perspective (`defteam`) — lower is better for the defense

In [ ]:
raw_off_epa = (
    pbp_filtered.groupby('posteam')['epa']
    .mean()
    .rename('raw_off_epa')
)

raw_def_epa = (
    pbp_filtered.groupby('defteam')['epa']
    .mean()
    .rename('raw_def_epa')
)

team_epa = pd.concat([raw_off_epa, raw_def_epa], axis=1).reset_index()
team_epa.columns = ['team', 'raw_off_epa', 'raw_def_epa']

print(f"League avg offensive EPA/play: {team_epa['raw_off_epa'].mean():.4f}")
print(f"League avg defensive EPA/play: {team_epa['raw_def_epa'].mean():.4f}")
team_epa.sort_values('raw_off_epa', ascending=False)

## Step 3: Build Game-Level Opponent Lookup

For each team-game, record who their opponent was. We'll use this to compute the average opponent unit EPA each team faced.

In [ ]:
# Get unique games with home/away teams
games_2025 = (
    pbp_filtered[['game_id', 'posteam', 'defteam']]
    .drop_duplicates()
)

# Build a lookup: for each (team, game), who was the opponent?
# From posteam perspective: posteam faced defteam's defense
off_matchups = games_2025[['posteam', 'defteam']].copy()
off_matchups.columns = ['team', 'opponent']

# From defteam perspective: defteam faced posteam's offense
def_matchups = games_2025[['defteam', 'posteam']].copy()
def_matchups.columns = ['team', 'opponent']

off_matchups = off_matchups.drop_duplicates()
def_matchups = def_matchups.drop_duplicates()

print(f"Offense matchup rows: {len(off_matchups)}")
print(f"Defense matchup rows: {len(def_matchups)}")

## Step 4: Opponent-Adjust EPA

**Formula:**
```
adj_off_epa = raw_off_epa - (avg_opp_def_epa - league_avg_def_epa)
adj_def_epa = raw_def_epa - (avg_opp_off_epa - league_avg_off_epa)
```

If your offense faced defenses that were better than average (lower raw_def_epa), we credit your offense upward. If your defense faced offenses that were worse than average, we penalize your defense downward.

In [ ]:
league_avg_off = team_epa['raw_off_epa'].mean()
league_avg_def = team_epa['raw_def_epa'].mean()

# Average defensive EPA of opponents each team's offense faced
off_matchups_epa = off_matchups.merge(
    team_epa[['team', 'raw_def_epa']],
    left_on='opponent', right_on='team'
).groupby('team_x')['raw_def_epa'].mean().rename('avg_opp_def_epa')

# Average offensive EPA of opponents each team's defense faced
def_matchups_epa = def_matchups.merge(
    team_epa[['team', 'raw_off_epa']],
    left_on='opponent', right_on='team'
).groupby('team_x')['raw_off_epa'].mean().rename('avg_opp_off_epa')

team_epa = team_epa.set_index('team')
team_epa = team_epa.join(off_matchups_epa).join(def_matchups_epa)

team_epa['adj_off_epa'] = team_epa['raw_off_epa'] - (team_epa['avg_opp_def_epa'] - league_avg_def)
team_epa['adj_def_epa'] = team_epa['raw_def_epa'] - (team_epa['avg_opp_off_epa'] - league_avg_off)

team_epa = team_epa.reset_index()
print("Adjusted EPA by team:")
team_epa[['team', 'raw_off_epa', 'adj_off_epa', 'raw_def_epa', 'adj_def_epa']].sort_values('adj_off_epa', ascending=False)

## Step 5: Load 2025 Schedule

Pull the full 2026 schedule (regular season only for now — playoffs aren't scheduled yet).

In [ ]:
schedule_2025 = nfl.import_schedules([2026])

# Keep regular season only
schedule_2025 = schedule_2025[schedule_2025['game_type'] == 'REG']

print(f"2025 regular season games: {len(schedule_2025)}")
print(f"Weeks: {schedule_2025['week'].min()} to {schedule_2025['week'].max()}")
schedule_2025[['week', 'home_team', 'away_team']].head(10)

## Step 6: Compute 2025 SOS Scores

For each team's 2025 opponents:
- **Offensive SOS** = average opponent `adj_def_epa` (lower = tougher schedule for your offense)
- **Defensive SOS** = average opponent `adj_off_epa` (higher = tougher schedule for your defense)

In [ ]:
# Build team-opponent pairs from 2026 schedule
home = schedule_2025[['home_team', 'away_team']].rename(columns={'home_team': 'team', 'away_team': 'opponent'})
away = schedule_2025[['away_team', 'home_team']].rename(columns={'away_team': 'team', 'home_team': 'opponent'})
matchups_2025 = pd.concat([home, away], ignore_index=True)

# Join with 2025 adjusted EPA
sos = matchups_2025.merge(
    team_epa[['team', 'adj_off_epa', 'adj_def_epa']],
    left_on='opponent', right_on='team',
    suffixes=('', '_opp')
)

sos_scores = sos.groupby('team').agg(
    off_sos=('adj_def_epa', 'mean'),   # avg opponent defensive EPA — lower = harder for your offense
    def_sos=('adj_off_epa', 'mean'),   # avg opponent offensive EPA — higher = harder for your defense
).reset_index()

sos_scores = sos_scores.sort_values('off_sos')
print("2025 SOS Scores (sorted by toughest offensive schedule):")
sos_scores

## Step 7: Sanity Check & Export

Quick look at the spread and export to CSV for further analysis.

In [ ]:
print("Offensive SOS range:")
print(f"  Hardest (lowest): {sos_scores['off_sos'].min():.4f} — {sos_scores.loc[sos_scores['off_sos'].idxmin(), 'team']}")
print(f"  Easiest (highest): {sos_scores['off_sos'].max():.4f} — {sos_scores.loc[sos_scores['off_sos'].idxmax(), 'team']}")

print("\nDefensive SOS range:")
print(f"  Hardest (highest): {sos_scores['def_sos'].max():.4f} — {sos_scores.loc[sos_scores['def_sos'].idxmax(), 'team']}")
print(f"  Easiest (lowest): {sos_scores['def_sos'].min():.4f} — {sos_scores.loc[sos_scores['def_sos'].idxmin(), 'team']}")

sos_scores.to_csv('sos_2025.csv', index=False)
print("\nExported to sos_2025.csv")